# 🛡️ PRDetect: Huấn luyện Đối kháng GATv2 trực tiếp trên Tập dữ liệu RAID

**Mục tiêu Phần 4:** Chứng minh Cơ chế Chú ý Đồ thị Động (**GATv2**) có khả năng học cách lọc bỏ các cạnh nhiễu và từ đối kháng trên tập dữ liệu **RAID**, vượt trội hơn **GCN2** tĩnh.

**Quy trình:**
1. Tải mẫu từ RAID (chứa cả văn bản người viết và văn bản AI qua các đòn tấn công đối kháng)
2. Xây dựng đồ thị cú pháp ngữ pháp (Dependency Tree) + RoBERTa node embeddings
3. Huấn luyện GCN2 (Baseline) và GATv2 (Cải tiến) trực tiếp trên dữ liệu RAID
4. Đánh giá và so sánh toàn diện (Accuracy, AUC-ROC, F1-Score)

## Bước 1: Cài đặt thư viện cần thiết

In [ ]:
# Cài đặt PyTorch Geometric, Transformers và Datasets
!pip install torch-geometric transformers datasets -q
!python -m spacy download en_core_web_sm -q

print("✅ Cài đặt hoàn tất!")


In [ ]:
# Kiểm tra môi trường GPU
import torch
import torch_geometric
print(f"PyTorch: {torch.__version__}")
print(f"PyG: {torch_geometric.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Bước 2: Tải và Chuẩn bị Dữ liệu Đối kháng từ RAID

In [ ]:
from datasets import load_dataset
import random
import pandas as pd

print("Đang lấy mẫu dữ liệu từ RAID (HuggingFace streaming)...")
TARGET_SAMPLES_PER_CLASS = 1500  # 1500 Human + 1500 Machine = 3000 mẫu tổng

raid_stream = load_dataset('liamdugan/raid', split='train', streaming=True)

human_texts = []
machine_texts = []
attack_stats = {}
model_stats = {}

for row in raid_stream:
    gen = row.get('generation', '').strip()
    if not gen or len(gen.split()) < 15 or len(gen.split()) > 250:
        continue
        
    model_name = row.get('model', 'unknown')
    attack_name = row.get('attack', 'none')
    
    if model_name == 'human' and len(human_texts) < TARGET_SAMPLES_PER_CLASS:
        human_texts.append(gen)
    elif model_name != 'human' and len(machine_texts) < TARGET_SAMPLES_PER_CLASS:
        machine_texts.append(gen)
        attack_stats[attack_name] = attack_stats.get(attack_name, 0) + 1
        model_stats[model_name] = model_stats.get(model_name, 0) + 1
        
    if len(human_texts) >= TARGET_SAMPLES_PER_CLASS and len(machine_texts) >= TARGET_SAMPLES_PER_CLASS:
        break

texts = human_texts + machine_texts
labels = [1] * len(human_texts) + [0] * len(machine_texts)  # 1: Human, 0: AI/Machine

# Trộn ngẫu nhiên
combined = list(zip(texts, labels))
random.seed(2024)
random.shuffle(combined)
texts, labels = zip(*combined)
texts, labels = list(texts), list(labels)

# Chia tập Train (70%), Val (15%), Test (15%)
n = len(texts)
n_tr, n_val = int(0.70 * n), int(0.15 * n)
train_texts, train_labels = texts[:n_tr], labels[:n_tr]
val_texts, val_labels = texts[n_tr:n_tr+n_val], labels[n_tr:n_tr+n_val]
test_texts, test_labels = texts[n_tr+n_val:], labels[n_tr+n_val:]

print(f"✅ Đã lấy thành công {len(texts)} mẫu RAID!")
print(f"  - Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")
print(f"  - Các đòn tấn công trong tập AI: {attack_stats}")
print(f"  - Các LLM models: {list(model_stats.keys())[:5]}")


## Bước 3: Xây dựng Đồ thị Cú pháp (Dependency Syntax Graphs + RoBERTa Embeddings)

In [ ]:
import spacy
import time
from transformers import RobertaTokenizer, RobertaModel
from tqdm.notebook import tqdm

print("Đang nạp mô hình SpaCy & RoBERTa...")
nlp = spacy.load("en_core_web_sm")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
roberta_model = RobertaModel.from_pretrained("roberta-base").to(device)
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base", do_lower_case=False)
roberta_model.eval()
print(f"✅ Models loaded on {device}")


In [ ]:
def build_graph_data(texts, labels, desc="Building graphs"):
    all_token_embeddings = []
    all_edge_index = []
    valid_labels = []
    errors = 0
    
    for idx, text in enumerate(tqdm(texts, desc=desc)):
        try:
            doc = nlp(text)
            tokenized_sentence = [token.text for token in doc]
            if len(tokenized_sentence) < 3:
                errors += 1
                continue
            
            # RoBERTa embeddings
            max_length = 512
            chunks = [tokenized_sentence[i:i+max_length] for i in range(0, len(tokenized_sentence), max_length)]
            chunk_outputs = []
            for chunk in chunks:
                token_ids = roberta_tokenizer.convert_tokens_to_ids(chunk)
                input_ids = torch.tensor(token_ids).unsqueeze(0).to(device)
                with torch.no_grad():
                    output = roberta_model(input_ids)
                chunk_outputs.append(output.last_hidden_state[0].cpu())
            token_embeddings = torch.cat(chunk_outputs, dim=0)
            
            # Cây ngữ pháp
            edge0, edge1 = [], []
            for word in doc:
                edge0.append(word.i)
                edge1.append(word.head.i)
            edge_index = torch.tensor([edge0, edge1], dtype=torch.long)
            
            all_token_embeddings.append(token_embeddings)
            all_edge_index.append(edge_index)
            valid_labels.append(labels[idx])
        except Exception:
            errors += 1
            
    y = torch.tensor(valid_labels, dtype=torch.float32)
    print(f"  {desc}: Thành công {len(valid_labels)}, Bỏ qua {errors}")
    return {'all_token_embeddings': all_token_embeddings, 'all_edge_index': all_edge_index, 'y': y}

t0 = time.time()
print("Bắt đầu dựng đồ thị cho tập dữ liệu RAID...")
raid_train_g = build_graph_data(train_texts, train_labels, "RAID Train")
raid_val_g   = build_graph_data(val_texts, val_labels, "RAID Val")
raid_test_g  = build_graph_data(test_texts, test_labels, "RAID Test")
print(f"✅ Hoàn tất trong {(time.time()-t0)/60:.1f} phút!")


## Bước 4: Định nghĩa Mô hình GCN2 (Baseline) và GATv2 (Cải tiến)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATv2Conv
from torch_geometric.data import Data

# ===== GCN2: Baseline (Chuẩn hóa bậc tĩnh) =====
class GCN2(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN2, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        self.fc = nn.Linear(output_dim, 1)
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.dropout(self.conv1(x, edge_index)))
        x = F.relu(self.dropout(self.conv2(x, edge_index)))
        x = self.fc(x)
        x = torch.mean(x, dim=0, keepdim=True)
        return torch.sigmoid(x)

# ===== GATv2: Cải tiến Phần 4 (Cơ chế chú ý động chống nhiễu) =====
class GATv2Model(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, heads=4, dropout=0.5):
        super(GATv2Model, self).__init__()
        self.conv1 = GATv2Conv(input_dim, hidden_dim, heads=heads, dropout=dropout, concat=True, add_self_loops=True)
        self.conv2 = GATv2Conv(hidden_dim * heads, output_dim, heads=1, dropout=dropout, concat=False, add_self_loops=True)
        self.fc = nn.Linear(output_dim, 1)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.elu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.elu(self.conv2(x, edge_index))
        x = self.dropout(x)
        x = self.fc(x)
        x = torch.mean(x, dim=0, keepdim=True)
        return torch.sigmoid(x)

print("✅ Đã định nghĩa GCN2 và GATv2!")


## Bước 5: Hàm Huấn luyện và Đánh giá

In [ ]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

def train_model(model, train_data, val_data, model_name, epochs=30, lr=0.0001, patience=5):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(dev)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.BCELoss()
    
    train_len = len(train_data['y'])
    val_len = len(val_data['y'])
    best_val_acc = -1
    pat_cnt = 0
    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    
    print(f"\n{'='*55}")
    print(f"  Huấn luyện {model_name} trên RAID | train={train_len}, val={val_len}")
    print(f"{'='*55}")
    
    t_start = time.time()
    for epoch in range(epochs):
        # Train
        model.train()
        e_loss, correct = 0.0, 0
        for i in range(train_len):
            data = Data(x=train_data['all_token_embeddings'][i],
                        edge_index=train_data['all_edge_index'][i],
                        y=train_data['y'][i]).to(dev)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y.float().view(-1, 1))
            loss.backward()
            optimizer.step()
            e_loss += loss.item()
            correct += ((out >= 0.5).long() == data.y.view(-1, 1)).sum().item()
            
        tr_loss = e_loss / train_len
        tr_acc = correct / train_len
        
        # Val
        model.eval()
        e_loss, correct = 0.0, 0
        with torch.no_grad():
            for i in range(val_len):
                data = Data(x=val_data['all_token_embeddings'][i],
                            edge_index=val_data['all_edge_index'][i],
                            y=val_data['y'][i]).to(dev)
                out = model(data)
                loss = criterion(out, data.y.float().view(-1, 1))
                e_loss += loss.item()
                correct += ((out >= 0.5).long() == data.y.view(-1, 1)).sum().item()
                
        v_loss = e_loss / val_len
        v_acc = correct / val_len
        
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(v_acc)
        
        tag = ""
        if v_acc >= best_val_acc:
            best_val_acc = v_acc
            pat_cnt = 0
            torch.save(model.state_dict(), f'/kaggle/working/raid_best_{model_name}.pth')
            tag = " ✓ BEST"
        else:
            pat_cnt += 1
            tag = f" ({pat_cnt}/{patience})"
            
        print(f"  Ep{epoch+1:2d}: tr_acc={tr_acc:.4f} v_acc={v_acc:.4f} tr_loss={tr_loss:.4f} v_loss={v_loss:.4f}{tag}")
        if pat_cnt >= patience:
            print(f"  → Dừng sớm (Early stopping)!")
            break
            
    print(f"  Thời gian: {time.time()-t_start:.0f}s | Best Val Acc: {best_val_acc:.4f}")
    model.load_state_dict(torch.load(f'/kaggle/working/raid_best_{model_name}.pth'))
    return model, history

def evaluate_model(model, test_data, model_name):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(dev)
    model.eval()
    test_len = len(test_data['y'])
    preds_prob = []
    correct = 0
    t_start = time.time()
    with torch.no_grad():
        for i in range(test_len):
            data = Data(x=test_data['all_token_embeddings'][i],
                        edge_index=test_data['all_edge_index'][i],
                        y=test_data['y'][i]).to(dev)
            out = model(data)
            preds_prob.append(out.item())
            correct += ((out >= 0.5).long() == data.y.view(-1, 1)).sum().item()
            
    y_pred = [1 if p >= 0.5 else 0 for p in preds_prob]
    y_true = test_data['y'].numpy().tolist()
    acc = correct / test_len
    auc = roc_auc_score(y_true, preds_prob)
    f1 = f1_score(y_true, y_pred)
    elapsed = time.time() - t_start
    print(f"  [{model_name}] Accuracy={acc:.4f} | AUC={auc:.4f} | F1={f1:.4f} | Time={elapsed:.1f}s")
    return {'model': model_name, 'acc': acc, 'auc': auc, 'f1': f1, 'time': elapsed}

print("✅ Ready!")


## Bước 6: Huấn luyện GCN2 (Baseline) trên RAID

In [ ]:
torch.manual_seed(2024)
torch.cuda.manual_seed_all(2024)

gcn_model = GCN2(768, 256, 64)
gcn_model, gcn_hist = train_model(gcn_model, raid_train_g, raid_val_g, "GCN2", epochs=30, lr=0.0001, patience=5)


## Bước 7: Huấn luyện GATv2 (Cải tiến Phần 4) trên RAID

In [ ]:
torch.manual_seed(2024)
torch.cuda.manual_seed_all(2024)

gatv2_model = GATv2Model(768, 64, 64, heads=4, dropout=0.5)
gatv2_model, gatv2_hist = train_model(gatv2_model, raid_train_g, raid_val_g, "GATv2", epochs=30, lr=0.0001, patience=5)


## Bước 8: Đánh giá và So sánh trên RAID Test Set

In [ ]:
print("="*60)
print("  📊 KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP ĐỐI KHÁNG RAID TEST")
print("="*60)

r_gcn = evaluate_model(gcn_model, raid_test_g, "GCN2 (Baseline)")
r_gatv2 = evaluate_model(gatv2_model, raid_test_g, "GATv2 (Cải tiến Phần 4)")

res_df = pd.DataFrame([r_gcn, r_gatv2])[['model', 'acc', 'auc', 'f1', 'time']]
res_df.columns = ['Mô hình', 'Accuracy', 'AUC-ROC', 'F1-Score', 'Inference(s)']
print("\n" + res_df.to_string(index=False, float_format='%.4f'))

diff_acc = (r_gatv2['acc'] - r_gcn['acc']) * 100
diff_auc = (r_gatv2['auc'] - r_gcn['auc']) * 100
diff_f1  = (r_gatv2['f1'] - r_gcn['f1']) * 100

print(f"\n🎯 ĐỘ TĂNG TRƯỞNG CỦA GATv2 SO VỚI GCN2:")
print(f"  - Accuracy: {diff_acc:+.2f}%")
print(f"  - AUC-ROC:  {diff_auc:+.2f}%")
print(f"  - F1-Score: {diff_f1:+.2f}%")

res_df.to_csv('/kaggle/working/raid_training_results.csv', index=False)
print("\n💾 Đã lưu kết quả: /kaggle/working/raid_training_results.csv")


## Bước 9: Vẽ Biểu đồ Quá trình Huấn luyện (Learning Curves)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(gcn_hist['val_acc'], label='GCN2 (Baseline) Val Acc', color='#e74c3c', linestyle='--')
axes[0].plot(gatv2_hist['val_acc'], label='GATv2 (Cải tiến) Val Acc', color='#2ecc71', linewidth=2)
axes[0].set_title('Validation Accuracy trên RAID', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(gcn_hist['val_loss'], label='GCN2 Val Loss', color='#e74c3c', linestyle='--')
axes[1].plot(gatv2_hist['val_loss'], label='GATv2 Val Loss', color='#2ecc71', linewidth=2)
axes[1].set_title('Validation Loss trên RAID', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('BCE Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/raid_learning_curves.png', dpi=300)
plt.show()
print("💾 Đã lưu biểu đồ: /kaggle/working/raid_learning_curves.png")
